Optimal Stopping & Signatures Implementation 


Signatures Computation

In [1]:
"""
signatures.py
-------------
A from-scratch implementation of the (truncated) signature of a path.

Everything here is built only from ideas you already know (integration,
vectors, dot products) plus ONE new rule: Chen's identity, which lets us
build a whole path's signature by multiplying the signatures of its
straight segments.

We represent a truncated signature as a dictionary:
    { 0: scalar,            # level 0 is always 1
      1: array shape (d,),  # level 1: the d first-order integrals
      2: array shape (d,d), # level 2: the d*d iterated integrals
      ... up to level N }
Each level `k` is a k-dimensional array with d entries per axis: the
number stored at position (i1,...,ik) is the iterated integral
S^{(i1,...,ik)} over the word (i1,...,ik).
"""

import numpy as np


# ---------------------------------------------------------------------------
# 1) Signature of a SINGLE straight segment.
#    Along a straight line the displacement is constant = `inc`, and the
#    level-k iterated integral is exactly inc^{⊗k} / k!  (see Step 2d).
# ---------------------------------------------------------------------------
def segment_signature(inc, N):
    inc = np.asarray(inc, dtype=float)
    sig = {0: np.array(1.0)}
    term = np.array(1.0)                 # will become inc^{⊗k} / k!
    for k in range(1, N + 1):
        term = np.tensordot(term, inc, axes=0) / k   # multiply by inc, divide by k
        sig[k] = term
    return sig


# ---------------------------------------------------------------------------
# 2) The tensor-algebra product (Definition 3.1.1 in the paper).
#    To get level m of the product a⊗b, sum over every split m = k + (m-k):
#        (a⊗b)_m = sum_{k=0}^{m}  a_k ⊗ b_{m-k}
#    This is the ONLY engine we need — Chen's identity is applied with it.
# ---------------------------------------------------------------------------
def tensor_mult(a, b, N):
    out = {}
    for m in range(N + 1):
        acc = None
        for k in range(m + 1):
            prod = np.tensordot(a[k], b[m - k], axes=0)  # outer product of the two levels
            acc = prod if acc is None else acc + prod
        out[m] = acc
    return out


# ---------------------------------------------------------------------------
# 3) Signature of a whole path, via Chen's identity.
#    A path is given as an array of shape (n_points, d): we connect the
#    points with straight lines, take each segment's signature, and
#    tensor-multiply them all together in order.
# ---------------------------------------------------------------------------
def path_signature(path, N):
    path = np.asarray(path, dtype=float)
    d = path.shape[1]
    sig = {0: np.array(1.0)}                  # start from the "empty path" signature = 1
    for lvl in range(1, N + 1):
        sig[lvl] = np.zeros((d,) * lvl)
    for i in range(len(path) - 1):
        seg = segment_signature(path[i + 1] - path[i], N)
        sig = tensor_mult(sig, seg, N)
    return sig


# ---------------------------------------------------------------------------
# 4) Helpers you'll want later.
# ---------------------------------------------------------------------------
def time_augment(x, T=1.0):
    """Turn a 1-D series x[0..n] into the 2-D augmented path (t, x_t).
    Time is added as coordinate 0 so the signature loses no information."""
    x = np.asarray(x, dtype=float)
    t = np.linspace(0.0, T, len(x))
    return np.column_stack([t, x])          # shape (n_points, 2)


def flatten(sig, N):
    """Flatten the level-dictionary into one plain vector, so that a linear
    functional <l, sig> is just an ordinary dot product l @ flatten(sig)."""
    parts = [np.atleast_1d(sig[k]).ravel() for k in range(N + 1)]
    return np.concatenate(parts)


def sig_length(d, N):
    """Length of the flattened truncated signature = 1 + d + d^2 + ... + d^N."""
    return sum(d ** k for k in range(N + 1))


# ---------------------------------------------------------------------------
# Quick self-check when run directly.
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # Straight line from a to b: signature level k must equal Delta^{⊗k}/k!
    a, b = np.array([0.0, 0.0]), np.array([3.0, 4.0])
    s = path_signature(np.vstack([a, b]), N=3)
    D = b - a
    assert np.allclose(s[1], D)
    assert np.allclose(s[2], np.tensordot(D, D, 0) / 2)

    # Chen's identity: splitting the segment does not change the signature.
    s2 = path_signature(np.vstack([a, (a + b) / 2, b]), N=3)
    assert all(np.allclose(s[k], s2[k]) for k in range(4))

    # Time-augment a little spread and print its flattened signature.
    x = [1.0, 1.2, 0.9, 0.7, 1.1]
    aug = time_augment(x, T=1.0)
    sig = path_signature(aug, N=2)
    print("augmented path shape:", aug.shape)
    print("flattened level<=2 signature:", flatten(sig, 2))
    print("expected length for d=2, N=2:", sig_length(2, 2))
    print("all checks passed.")

augmented path shape: (5, 2)
flattened level<=2 signature: [ 1.      1.      0.1     0.5     0.1375 -0.0375  0.005 ]
expected length for d=2, N=2: 7
all checks passed.


Weights l Computation